# Day 8 — Conversational AI, System Prompts & Multi-Shot Prompting

## Future Revision Notes

This notebook uses the **original instructor Day 3 learning notebook as the primary source**.

Your separate practice notebook was treated only as evidence of your practice. Its Gemini approach is represented below so that the same lesson can be recalled using **both OpenAI and Gemini**.

## Today's core ideas

### 1. Conversational AI

A conversational assistant receives:

```text
current user message
        +
previous conversation history
        +
system instructions
        ↓
      LLM
        ↓
     response
```

The important application pattern is:

```python
chat(message, history)
```

`message` = the user's current message.

`history` = the previous conversation supplied by the chat UI.

### 2. Conversation history

For the OpenAI-compatible `messages` format:

```python
messages = (
    [{"role": "system", "content": system_message}]
    + history
    + [{"role": "user", "content": message}]
)
```

This means:

```text
system instruction
      ↓
previous messages
      ↓
current user message
```

The model can only use the previous conversation if your application actually sends that history.

### 3. System prompt

A system message provides high-level instructions, context, behavior, or constraints.

Example:

```python
system_message = "You are a helpful assistant"
```

For a business assistant, the system message can describe:

- the business
- the desired tone
- important information
- rules
- examples of desired responses

### 4. One-shot / multi-shot prompting

The lesson demonstrates giving the model an example of the desired answer inside the system instructions.

**One-shot:**

```text
Instruction
Example input → example style/output
New input
```

**Multi-shot:**

```text
Instruction
Example 1
Example 2
Example 3
New input
```

The examples help the model infer the desired pattern.

### 5. Streaming

Instead of waiting for the complete answer:

```python
stream = client.chat.completions.create(
    ...,
    stream=True
)
```

the application receives chunks.

Core pattern:

```python
response = ""

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    yield response
```

`yield` lets Gradio display the progressively growing response.

### 6. Conditional system instructions

The lesson also shows that the system message can be modified depending on the user's current request.

Example:

```python
relevant_system_message = system_message

if "belt" in message.lower():
    relevant_system_message += (
        " The store does not sell belts; point out other items on sale."
    )
```

Then the modified system instruction is sent for that request.

### 7. First look at RAG

The lesson introduces the idea of giving an LLM useful external context.

A simple mental model is:

```text
User question
     ↓
Find relevant information
     ↓
Put relevant information into the prompt/context
     ↓
LLM
     ↓
Answer
```

The important idea for today's lesson is **context**: the model performs better when the relevant information is supplied to it.

## Easy memory trick

```text
Chatbot       → message + history
System prompt → behavior/context/rules
One-shot      → one example
Multi-shot    → multiple examples
Streaming     → chunks + yield
RAG           → retrieve context + give it to the LLM
```

# OpenAI — Instructor's Conversational Chat Pattern

The original notebook uses the OpenAI Python client and Chat Completions.

The key API structure is:

```python
client.chat.completions.create(
    model=MODEL,
    messages=messages
)
```

For a conversational Gradio callback:

```python
def chat(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content
```

The original lesson uses:

```python
gr.ChatInterface(fn=chat, type="messages")
```

so Gradio supplies the current message and conversation history to the callback.

In [ ]:
# OpenAI conversational chatbot — same core pattern as the instructor notebook

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

openai = OpenAI(api_key=openai_api_key)

MODEL = "gpt-4.1-mini"

system_message = "You are a helpful assistant"

def chat_openai(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content

# Run when you want to launch the UI:
# gr.ChatInterface(fn=chat_openai, type="messages").launch()

# Gemini — Same Conversational Concept

The same lesson can be implemented with Google's native Gemini client.

The important difference is the SDK syntax, not the underlying idea.

```text
OpenAI
  system + history + current message
             ↓
      chat.completions.create()

Gemini
  system instruction + history + current message
             ↓
      models.generate_content()
```

The Gemini client keeps the system instruction in the generation configuration and the conversation content in `contents`.

The exact Gemini model name should be replaced with a model currently available to your API key.

In [ ]:
# Gemini native SDK — conversational chatbot

import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
import gradio as gr

load_dotenv(override=True)

google_api_key = os.getenv("GEMINI_API_KEY")

gemini = genai.Client(api_key=google_api_key)

GEMINI_MODEL = "gemini-3.5-flash-lite"

system_message = "You are a helpful assistant"

def chat_gemini(message, history):
    contents = []

    for h in history:
        role = "user" if h["role"] == "user" else "model"
        contents.append(
            types.Content(
                role=role,
                parts=[types.Part.from_text(text=h["content"])]
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    )

    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_message
        )
    )

    return response.text

# Run when you want to launch the UI:
# gr.ChatInterface(fn=chat_gemini, type="messages").launch()

# OpenAI-Compatible Gemini — Alternative API Style

Your practice also used the OpenAI Python client with Gemini's OpenAI-compatible endpoint.

This is useful because the **client syntax looks like the OpenAI Chat Completions syntax**, while the request is sent to Gemini.

```python
from openai import OpenAI

gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
```

Then the conversational pattern is:

```python
messages = (
    [{"role": "system", "content": system_message}]
    + history
    + [{"role": "user", "content": message}]
)

response = gemini.chat.completions.create(
    model=GEMINI_MODEL,
    messages=messages
)
```

### Remember the difference

```text
Native Gemini SDK
    ↓
google.genai
    ↓
gemini.models.generate_content(...)

OpenAI-compatible Gemini endpoint
    ↓
openai Python client
    ↓
gemini.chat.completions.create(...)
```

Both are ways of calling Gemini; the SDK/API style is different.

In [ ]:
# Gemini through Google's OpenAI-compatible endpoint

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

google_api_key = os.getenv("GEMINI_API_KEY")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_compatible = OpenAI(
    api_key=google_api_key,
    base_url=gemini_url
)

GEMINI_MODEL = "gemini-3.5-flash-lite"

system_message = "You are a helpful assistant"

def chat_gemini_compatible(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = gemini_compatible.chat.completions.create(
        model=GEMINI_MODEL,
        messages=messages
    )

    return response.choices[0].message.content

# Run when you want:
# gr.ChatInterface(
#     fn=chat_gemini_compatible,
#     title="Gemini Chatbot",
#     type="messages"
# ).launch()

# Streaming — OpenAI and Gemini-Compatible Chat API

The instructor notebook's streaming pattern is:

```python
stream = openai.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=True
)

response = ""

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    yield response
```

The same pattern works with the OpenAI-compatible Gemini client because it uses the same Chat Completions interface.

The important concept is:

```text
stream=True
    ↓
chunks arrive
    ↓
chunk.choices[0].delta.content
    ↓
append to response
    ↓
yield response
    ↓
Gradio updates the chat
```

In [ ]:
# OpenAI streaming

def stream_openai(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
# Gemini through the OpenAI-compatible endpoint — streaming

def stream_gemini_compatible(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    stream = gemini_compatible.chat.completions.create(
        model=GEMINI_MODEL,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

# System Prompt + One-Shot Prompting

The original lesson changes the system message into a business-specific instruction.

Example from the lesson:

```text
You are a helpful assistant in a clothes store.
Encourage customers to try items that are on sale.
Hats are 60% off and most other items are 50% off.
```

It also gives an example response. That example is the **one-shot** part.

### General structure

```text
System instruction
      +
Example of desired behavior
      +
Conversation history
      +
Current user message
```

The model uses the supplied example to infer how it should respond.

### Multi-shot

If you provide multiple examples:

```text
Example 1
Example 2
Example 3
Current input
```

that is commonly called **few-shot** or **multi-shot prompting**.

In [ ]:
# OpenAI — business system prompt with an example

system_message = (
    "You are a helpful assistant in a clothes store. "
    "Gently encourage customers to try items that are on sale. "
    "Hats are 60% off, and most other items are 50% off. "
    "For example, if the customer says "
    "'I'm looking to buy a hat', you could reply "
    "'Wonderful - we have lots of hats - including several that are part "
    "of our sales event.' Encourage customers to buy hats if they are "
    "unsure what to get."
)

def chat_openai_store(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# Gemini native SDK — same system prompt + example idea

def chat_gemini_store(message, history):
    contents = []

    for h in history:
        role = "user" if h["role"] == "user" else "model"
        contents.append(
            types.Content(
                role=role,
                parts=[types.Part.from_text(text=h["content"])]
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    )

    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_message
        )
    )

    return response.text

# Conditional System Instructions

The original lesson adds extra context only when the customer asks about belts.

The pattern is:

```python
relevant_system_message = system_message

if "belt" in message.lower():
    relevant_system_message += " ...extra instruction..."

messages = (
    [{"role": "system", "content": relevant_system_message}]
    + history
    + [{"role": "user", "content": message}]
)
```

This is useful when some instructions are relevant only to particular requests.

In [ ]:
# OpenAI — conditional system instruction

def chat_openai_conditional(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    relevant_system_message = system_message

    if "belt" in message.lower():
        relevant_system_message += (
            " The store does not sell belts; if asked for belts, "
            "point out other items on sale."
        )

    messages = (
        [{"role": "system", "content": relevant_system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# Gemini native SDK — conditional system instruction

def chat_gemini_conditional(message, history):
    contents = []

    for h in history:
        role = "user" if h["role"] == "user" else "model"
        contents.append(
            types.Content(
                role=role,
                parts=[types.Part.from_text(text=h["content"])]
            )
        )

    contents.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    )

    relevant_system_message = system_message

    if "belt" in message.lower():
        relevant_system_message += (
            " The store does not sell belts; if asked for belts, "
            "point out other items on sale."
        )

    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=relevant_system_message
        )
    )

    return response.text

# First Look at RAG

RAG = **Retrieval-Augmented Generation**.

The basic idea introduced here is that the application can provide relevant external information to the LLM instead of relying only on information already contained in the model.

```text
User question
      ↓
Retrieve relevant information
      ↓
Put information into context
      ↓
LLM
      ↓
Answer
```

For future lessons, this becomes a more complete retrieval pipeline using documents, embeddings/vector search, and context injection.

For today's recall, remember only:

> **RAG = retrieve relevant context, then give that context to the LLM to help generate the answer.**

# Final Revision Checklist

Before moving on, I should be able to explain:

- [ ] What `chat(message, history)` means in a Gradio chatbot.
- [ ] Why conversation history is included in the next LLM request.
- [ ] How `system`, `user`, and `assistant/model` messages differ.
- [ ] How to build `messages` with `system + history + current user`.
- [ ] What a system prompt is used for.
- [ ] What one-shot prompting means.
- [ ] What multi-shot/few-shot prompting means.
- [ ] What `stream=True` does.
- [ ] Why `yield` is used for streaming.
- [ ] How conditional system instructions work.
- [ ] The basic idea behind RAG.
- [ ] The difference between OpenAI's client and Gemini's native SDK.
- [ ] How Gemini can also be accessed through an OpenAI-compatible endpoint.

# Day 3 - Conversational AI - aka Chatbot!

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [ ]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [ ]:
def chat(message, history):
    return "bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## OK! Let's write a slightly better chat callback!

In [ ]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [ ]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>